# Version 3 — ChromaDB RAG
**Project:** AI Placement Preparation Assistant using RAG

V3 uses **ChromaDB**, a vector database that stores vectors, text and metadata together and supports persistent storage and metadata filtering.

**Colab secret needed:** `Gemini_API`

| # | Roadmap stage | Covered in this notebook |
|---|---|---|
| 1 | Document Loading | ✅ `PyPDFLoader` + category tag |
| 2 | Text Splitting / Chunking | ✅ `RecursiveCharacterTextSplitter` |
| 3 | Embeddings | ✅ Gemini embeddings |
| 4 | Vector Database | ✅ **ChromaDB** (persistent) |
| 5 | Retrieval | ✅ Similarity search + metadata filter |
| 6 | LLM / Gemini | ✅ Gemini chat model |
| 7 | Answer + Source/Page Citation | ✅ File + page + category |
| 8 | Streamlit App | Later (`app.py`) |

### Important fix in this version
The previous notebook used `gemini-3.6-flash` for the final answer. Your saved run shows a **429 RESOURCE_EXHAUSTED** error because that Gemini API project had exhausted its free-tier request quota.

This fixed notebook:
1. Uses a configurable `LLM_MODEL` variable.
2. Uses `gemini-3.5-flash-lite` by default.
3. Handles Gemini 429/quota errors without crashing the notebook.
4. Removes the automatic model call from the notebook startup.
5. Keeps ChromaDB, metadata filtering, retrieval and source/page citation logic intact.

**Important:** changing the model does not magically restore an exhausted API quota. If the selected model/project is also out of quota, the notebook will clearly report that and you must wait for the reset or use a project/key with available quota.


## Step 1 — Setup
`langchain-chroma` connects LangChain to ChromaDB.

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-google-genai langchain-chroma chromadb pypdf python-dotenv

## Step 2 — Gemini API key

In [ ]:
import os
from getpass import getpass

api_key = None

# 1) Google Colab: secret named Gemini_API  (left sidebar -> key icon)
try:
    from google.colab import userdata
    api_key = userdata.get("Gemini_API")
except Exception:
    pass

# 2) VS Code / local: a .env file in the project folder (GOOGLE_API_KEY=...) or an environment variable
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# 3) Last option: type it (the text is hidden)
if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("No Gemini API key was provided.")

os.environ["GOOGLE_API_KEY"] = api_key
print("API key loaded:", bool(os.environ.get("GOOGLE_API_KEY")))


## Step 3 — Check PDFs

In [ ]:
import os

from pathlib import Path

# Works in Google Colab (PDFs uploaded to /content) AND in VS Code (project folder with data/).
if Path("/content/hr_questions.pdf").exists():
    DATA_DIR = "/content"
    WORK_DIR = "/content"                      # where saved indexes go in Colab
else:
    _here = Path.cwd()
    _project = next((p for p in [_here, _here.parent] if (p / "data" / "hr_questions.pdf").exists()), _here)
    DATA_DIR = str(_project / "data")
    WORK_DIR = str(_project / "vectorstores")  # saved indexes go here (ignored by git)
os.makedirs(WORK_DIR, exist_ok=True)
print("PDF folder :", DATA_DIR)

PDF_FILES = [
    f"{DATA_DIR}/technical_interview_.pdf",
    f"{DATA_DIR}/resume_guidelines.pdf",
    f"{DATA_DIR}/hr_questions.pdf",
    f"{DATA_DIR}/aptitude.pdf",
]

missing = [f for f in PDF_FILES if not os.path.exists(f)]

if missing:
    print("❌ These files were not found:")
    for f in missing:
        print("   ", f)
    print("Upload them to Colab (left sidebar → Files) and run this cell again.")
else:
    print("✅ All 4 PDFs found")

## Step 4 — Load PDFs
📍 **Roadmap 1: Document Loading**

In [4]:
from langchain_community.document_loaders import PyPDFLoader

all_documents = []

for path in PDF_FILES:
    loader = PyPDFLoader(path)
    docs = loader.load()                      # one Document per page
    print(f"{os.path.basename(path):28s} -> {len(docs)} page(s)")
    all_documents.extend(docs)

print("\nTotal Documents (pages):", len(all_documents))

/tmp/ipykernel_631/1712772264.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


technical_interview_.pdf     -> 5 page(s)
resume_guidelines.pdf        -> 3 page(s)
hr_questions.pdf             -> 3 page(s)
aptitude.pdf                 -> 4 page(s)

Total Documents (pages): 15


## Step 5 — Add custom metadata (category)
**What:** we add our own key `category` to each page's metadata.

**Why:** loaders only give `source` and `page`. By adding `category`, the user (or app) can say *"search only HR questions"*. This is called **metadata enrichment** and is very common in real RAG systems.

> Only the four categories of **your** project are used.

In [5]:
# Our project has exactly 4 kinds of material. We label every page with its category.
CATEGORY_BY_FILE = {
    "technical_interview_.pdf": "technical",
    "resume_guidelines.pdf":    "resume",
    "hr_questions.pdf":         "hr",
    "aptitude.pdf":             "aptitude",
}

for doc in all_documents:
    file_name = os.path.basename(doc.metadata["source"])
    doc.metadata["category"] = CATEGORY_BY_FILE.get(file_name, "other")

print(all_documents[0].metadata)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-08T11:57:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-08T11:57:06+00:00', 'subject': '(unspecified)', 'title': 'Technical Interview Preparation Notes: AI, Machine Learning and NLP', 'trapped': '/False', 'source': '/content/technical_interview_.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'category': 'technical'}


## Step 6 — Split into chunks
📍 **Roadmap 2** — chunks inherit the metadata, including `category`.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,      # max characters in one chunk
    chunk_overlap=100,   # characters repeated between neighbouring chunks
)

chunks = text_splitter.split_documents(all_documents)

print("Pages  :", len(all_documents))
print("Chunks :", len(chunks))
print("\n--- Chunk 1 ---")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Pages  : 15
Chunks : 53

--- Chunk 1 ---
Technical Interview Preparation Notes: AI,
 Machine Learning and NLP
Section 1: Artificial Intelligence and Machine Learning Basics
What is Artificial Intelligence?
Artificial Intelligence is the field of building systems that perform tasks that normally need human
intelligence, such as understanding language, recognising images, making decisions and learning
from experience. Machine learning and deep learning are subsets of AI.
What is the difference between AI, Machine Learning and Deep Learning?
AI is the broad goal of making machines intelligent. Machine Learning is a subset of AI where
systems learn patterns from data instead of being explicitly programmed. Deep Learning is a subset

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-08T11:57:06+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-08T11:57:06+00:00', 'subject': '(unspecified)', 'title': 'Techni

## Step 7 — Embeddings
📍 **Roadmap 3**

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Current Gemini embedding model.
EMBEDDING_MODEL = "gemini-embedding-001"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL,
)

# Test one embedding request.
try:
    vector = embeddings.embed_query("What is overfitting?")
    print("Embedding model:", EMBEDDING_MODEL)
    print("Vector length  :", len(vector))
    print("First 5 numbers:", vector[:5])
except Exception as e:
    print("\n❌ Embedding test failed.")
    print(type(e).__name__, ":", e)
    print("\nIf this is a 429/quota error, the Gemini API quota for embeddings has been reached.")


Embedding model: gemini-embedding-001
Vector length  : 3072
First 5 numbers: [-0.006956061, -0.00035724926, 0.0017957747, -0.052621245, 0.009199971]


## Step 8 — Create the Chroma database
📍 **Roadmap 4: Vector Database**

- `collection_name` → like a table name inside Chroma
- `persist_directory` → folder where data is saved on disk

We delete the old folder first. If we ran `from_documents` twice on the same folder, every chunk would be **added twice** (duplicates).

In [ ]:
import shutil
from langchain_chroma import Chroma

CHROMA_DIR = os.path.join(WORK_DIR, "chroma_db")
COLLECTION = "placement_notes"

shutil.rmtree(CHROMA_DIR, ignore_errors=True)   # start clean

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
)

print("✅ Chroma created")
print("Chunks stored:", vectorstore._collection.count())
print("Folder contents:", os.listdir(CHROMA_DIR))

## Step 9 — Reload from disk (proves persistence)

In [9]:
vectorstore = Chroma(
    collection_name=COLLECTION,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)
print("✅ Reloaded. Chunks:", vectorstore._collection.count())

✅ Reloaded. Chunks: 53


**Observe:** no `save_local` call was needed — Chroma saved automatically.

## Step 10 — Retrieval
📍 **Roadmap 5: Retrieval**

Normal search first (all documents).

In [10]:
question = "How should I describe a project on my resume?"

results = vectorstore.similarity_search_with_score(question, k=4)
for doc, score in results:
    print(f"distance={score:.4f} | {doc.metadata['category']:9s} | {os.path.basename(doc.metadata['source'])} | page {doc.metadata['page'] + 1}")

distance=0.4217 | resume    | resume_guidelines.pdf | page 2
distance=0.4782 | resume    | resume_guidelines.pdf | page 2
distance=0.5329 | resume    | resume_guidelines.pdf | page 3
distance=0.6205 | resume    | resume_guidelines.pdf | page 1


### Metadata filtering
Now the **same question**, but only inside one category. This is the big advantage of Chroma.

Try changing `"resume"` to `"hr"` and see how results change.

In [11]:
filtered = vectorstore.similarity_search_with_score(
    "How should I prepare for the interview?",
    k=3,
    filter={"category": "hr"},        # only HR material
)

for doc, score in filtered:
    print(f"distance={score:.4f} | {doc.metadata['category']} | page {doc.metadata['page'] + 1}")
    print("   ", doc.page_content[:120].replace("\n", " "), "\n")

distance=0.5371 | hr | page 3
    phone to silent. What questions should I ask the interviewer? Ask about the team and role, the training and learning opp 

distance=0.5483 | hr | page 3
    Section 3: Interview Etiquette and Preparation How should I prepare for an HR round? Research the company, review the jo 

distance=0.5743 | hr | page 1
    Connect your skills and projects to what the company needs. Briefly state your relevant technical skills, your ability t 



## Step 11 — Gemini LLM
This is the generation stage: retrieved chunks are sent to Gemini to produce the final answer.

The model name is kept in one variable (`LLM_MODEL`) so it can be changed easily if your Gemini API project has a different model/quota available.

**Important:** a 429 `RESOURCE_EXHAUSTED` error is an API quota problem, not a ChromaDB retrieval problem.


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Keep the model in one variable so it is easy to change.
# 3.5 Flash-Lite is a cost-efficient model suitable for this RAG project.
LLM_MODEL = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    temperature=0,
)

print("LLM:", LLM_MODEL)


LLM: gemini-3.5-flash-lite


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

NOT_FOUND = "I could not find the answer in the uploaded documents."

prompt = ChatPromptTemplate.from_template("""You are an AI Placement Preparation Assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, reply exactly:
"{not_found}"
Do not use outside knowledge.

Context:
{context}

Question: {question}

Answer:""")

# prompt -> llm -> plain text
answer_chain = prompt | llm | StrOutputParser()

import time


def invoke_with_retry(payload, tries=3):
    """Call Gemini; wait and retry when the model is only busy (503)."""
    for i in range(tries):
        try:
            return answer_chain.invoke(payload)
        except Exception as e:
            if ("503" in str(e) or "UNAVAILABLE" in str(e)) and i < tries - 1:
                print("⏳ Gemini is busy (503). Retrying...")
                time.sleep(3 * (i + 1))
                continue
            raise


In [14]:
def format_context(docs):
    """Join retrieved chunks into one text block for the prompt."""
    return "\n\n".join(d.page_content for d in docs)


def format_sources(docs):
    """Unique 'file - page' list. PyPDFLoader pages start at 0, humans start at 1."""
    seen, lines = set(), []
    for d in docs:
        name = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page", 0) + 1
        key = (name, page)
        if key not in seen:
            seen.add(key)
            lines.append(f"{name} - page {page}")
    return lines

## Step 12 — RAG function with optional category filter
📍 **Roadmap 5 + 6 + 7**

In [ ]:
# Chroma distance (L2): smaller = more similar. In V2 we measured ~0.45-0.61 for questions that are
# in the PDFs and ~0.85-0.94 for questions that are not, so 0.73 sits in the gap.
# (1.2 was too high: it would never refuse anything.)
SCORE_THRESHOLD = 0.73

def ask(question, k=4, category=None):
    search_filter = {"category": category} if category else None

    try:
        results = vectorstore.similarity_search_with_score(
            question,
            k=k,
            filter=search_filter
        )
    except Exception as e:
        print("❌ Retrieval failed.")
        print(type(e).__name__, ":", e)
        return

    if not results:
        print("Question:", question)
        print("\nAnswer:\n", NOT_FOUND)
        return

    # Chroma distance: smaller generally means more similar.
    docs = [d for d, score in results if score <= SCORE_THRESHOLD]

    if not docs:
        print("Question:", question)
        print("\nAnswer:\n", NOT_FOUND)
        print("\nNo retrieved chunk passed the similarity threshold.")
        return

    try:
        answer = invoke_with_retry({
            "context": format_context(docs),
            "question": question,
            "not_found": NOT_FOUND,
        })
    except Exception as e:
        print("Question:", question)
        print("\n❌ Gemini could not generate the answer.")
        print(type(e).__name__, ":", e)

        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print("\nThis is a Gemini API quota/rate-limit error.")
            print("Wait for the quota reset or use a Gemini API project/key with available quota.")
        return

    print("Question:", question, f"(category filter: {category})")
    print("\nAnswer:\n", answer)

    if NOT_FOUND not in answer:
        print("\nSources:")
        seen = set()
        for d in docs:
            key = (d.metadata["source"], d.metadata["page"])
            if key in seen:               # show each file + page only once
                continue
            seen.add(key)
            print(
                f" - [{d.metadata['category']}] "
                f"{os.path.basename(d.metadata['source'])} "
                f"- page {d.metadata['page'] + 1}"
            )


In [ ]:
# Test 1 — question that should be answered from the technical PDF
ask("What is a confusion matrix?")

print("\n" + "=" * 70 + "\n")

# Test 2 — resume category filter
ask("What are common resume mistakes?", category="resume")

print("\n" + "=" * 70 + "\n")

# Test 3 — aptitude category filter
ask("How do I solve time and work problems?", category="aptitude")

print("\n" + "=" * 70 + "\n")

# Test 4 — unsupported question
ask("What is the capital of Australia?")


## Step 13 — Interactive chat (optional)

Run the next cell only when you want to chat manually.

You can optionally filter by starting the question with:
`hr:`, `technical:`, `resume:` or `aptitude:`.

Example:
`hr: What is the STAR method?`

Type `exit` or `quit` to stop.


In [17]:
VALID = {"hr", "technical", "resume", "aptitude"}

print("Interactive chat started. Type 'exit' to stop.")

while True:
    q = input("\nYou: ").strip()

    if q.lower() in ["exit", "quit"]:
        print("Goodbye!")
        break

    if not q:
        continue

    category = None

    if ":" in q:
        prefix, remainder = q.split(":", 1)
        if prefix.strip().lower() in VALID:
            category = prefix.strip().lower()
            q = remainder.strip()

    ask(q, category=category)


Interactive chat started. Type 'exit' to stop.

You: exit
Goodbye!


## 🎓 Viva corner

**Q. What is ChromaDB?**

An open-source vector database. It stores embeddings, the original text and metadata together and saves them on disk.

**Q. FAISS vs ChromaDB?**

FAISS = fast index library, we manage saving ourselves. Chroma = full database with automatic persistence and metadata filtering.

**Q. What is a collection?**

A named group of vectors/documents inside Chroma, similar to a table in SQL.

**Q. What is metadata filtering and why use it?**

Restricting search to chunks whose metadata matches, e.g. category = hr. It makes results more precise and faster.

**Q. What happens if you run `from_documents` twice on the same folder?**

Duplicates are added. That is why we delete the folder before rebuilding.

**Q. Where does `category` come from?**

The loader does not give it. We added it ourselves from the file name — metadata enrichment.

